# HCP Grain Boundary Generator — Getting Started

This notebook demonstrates how to:
1. Enumerate available CSL grain boundaries for any c/a ratio
2. Query and filter by Sigma, rotation axis, or angle
3. Build twist and tilt grain boundary structures with ASE
4. Handle non-ideal c/a metals (enumerate at rational, rescale to real)
5. Orthogonalize simulation cells
6. Export structures

In [ ]:
from math import sqrt
from hcp_gb_generator import (
    enumerate_hcp_csl,
    enumerate_0001_csl,
    find_csl,
    print_csl_table,
    to_dataframe,
    build_gb,
    build_gb_rescaled,
    build_twist_gb,
    build_tilt_gb,
    csl_slab_directions,
    rescale_to_lattice,
    orthogonalize_gb_plane,
    find_orthogonal_cell,
)

## 1. Enumerate CSL grain boundaries

### 1a. [0001]-axis GBs (c/a independent, fast)

These are basal-plane twist boundaries.  The available Sigma values
depend only on the 2D triangular lattice geometry and are the same
for **any** HCP material.

In [ ]:
res_0001 = enumerate_0001_csl(sigma_max=50)
print_csl_table(res_0001)

### 1b. Full enumeration (all axes, c/a dependent)

Tilt GBs with rotation axes in the basal plane depend on the c/a ratio.
Common values:
- Ti:  c/a ≈ 1.587
- Mg:  c/a ≈ 1.624
- Ideal HCP: c/a = sqrt(8/3) ≈ 1.633
- Zn:  c/a ≈ 1.856

In [ ]:
IDEAL_CA = sqrt(8 / 3)

all_results = enumerate_hcp_csl(
    ca_ratio=IDEAL_CA,
    sigma_max=25,
    max_idx=3,       # max Miller index for axis search (3 is fast, 9 for thorough)
)

print(f"Found {len(all_results)} CSL grain boundaries")
print_csl_table(all_results)

### 1c. As a pandas DataFrame

In [ ]:
df = to_dataframe(all_results)
df.head(15)

## 2. Query and filter

### 2a. Find all GBs for a specific Sigma

In [ ]:
sigma7 = find_csl(all_results, sigma=7)
for r in sigma7:
    print(f"  Sigma={r['sigma']}  disorientation={r['disorientation_angle']:.2f}  axis={r['axis_miller']}")

### 2b. Filter by rotation axis family

Use 4-index Miller-Bravais notation.  The filter is hex-symmetry-aware,
so `[1,0,-1,0]` matches all `<10-10>` family members.

In [ ]:
# [0001] axis — basal twist boundaries
print("=== [0001] axis ===")
for r in find_csl(all_results, axis_miller_bravais=[0, 0, 0, 1], ca_ratio=IDEAL_CA):
    print(f"  Sigma={r['sigma']:>3}  disorientation={r['disorientation_angle']:>7.2f}")

# [10-10] axis — prism tilt boundaries
print("\n=== [10-10] axis ===")
for r in find_csl(all_results, axis_miller_bravais=[1, 0, -1, 0], ca_ratio=IDEAL_CA):
    print(f"  Sigma={r['sigma']:>3}  disorientation={r['disorientation_angle']:>7.2f}")

# [11-20] axis — a-direction tilt boundaries
print("\n=== [11-20] axis ===")
for r in find_csl(all_results, axis_miller_bravais=[1, 1, -2, 0], ca_ratio=IDEAL_CA):
    print(f"  Sigma={r['sigma']:>3}  disorientation={r['disorientation_angle']:>7.2f}")

### 2c. Filter by angle range

In [ ]:
low_angle = find_csl(all_results, angle_min=10, angle_max=30)
print(f"Found {len(low_angle)} GBs between 10-30 degrees:")
for r in low_angle:
    print(f"  Sigma={r['sigma']:>3}  disorientation={r['disorientation_angle']:>7.2f}  axis={r['axis_miller']}")

### 2d. Generate on-the-fly for a different c/a ratio

In [ ]:
# No pre-computed results needed — find_csl generates them
ti_sigma7 = find_csl(ca_ratio=1.587, sigma=7, sigma_max=10)
print(f"Ti (c/a=1.587): {len(ti_sigma7)} Sigma=7 boundaries")
for r in ti_sigma7:
    print(f"  disorientation={r['disorientation_angle']:.2f}  axis={r['axis_miller']}")

## 3. Build grain boundary structures

### 3a. [0001] Twist grain boundary

Both grains share the (0001) surface.  The upper grain is rotated
in-plane by the CSL angle.

In [ ]:
# Titanium lattice parameters
a_Ti, c_Ti = 2.95, 4.68

# Find Sigma=7 [0001] twist
rec = find_csl(ca_ratio=c_Ti / a_Ti, sigma=7,
               axis_miller_bravais=[0, 0, 0, 1], sigma_max=10)[0]

# Build the bicrystal (fully periodic → 2 GBs per cell)
twist_gb = build_twist_gb(
    rec,
    element="Ti",
    a=a_Ti,
    c=c_Ti,
    n_layers=4,
    interface_distance=0.5,  # spacing at GB plane (Å)
)

print(f"Sigma = {twist_gb.info['sigma']}")
print(f"Disorientation = {twist_gb.info['disorientation_angle']:.2f} deg")
print(f"Atoms = {len(twist_gb)}")
print(f"Cell  = {twist_gb.cell.lengths().round(2)}")
print(f"PBC   = {list(twist_gb.pbc)}")
print(f"Grains: {sorted(set(twist_gb.arrays['grain_id']))}")

### 3b. Symmetric tilt grain boundary

Rotation axis lies in the GB plane.  The two grains are
mirror images across the boundary and **share a coincident
atomic layer** at the interface.  By default `merge_boundary_layer=True`
removes this duplicate.

In [ ]:
# Tilt GBs are c/a-dependent — use ideal c/a where many are known
# (Ti's non-ideal c/a produces fewer tilt CSLs at low Sigma)
ca_ideal = sqrt(8 / 3)
a_ideal, c_ideal = 2.95, 2.95 * ca_ideal  # scale to Ti-like size

all_res = enumerate_hcp_csl(ca_ratio=ca_ideal, sigma_max=20, max_idx=3)
rec_tilt = find_csl(all_res, sigma=17,
                    axis_miller_bravais=[1, 1, -2, 0],
                    ca_ratio=ca_ideal)[0]

tilt_gb = build_tilt_gb(
    rec_tilt,
    element="Ti",
    a=a_ideal,
    c=c_ideal,
    n_layers=4,
    interface_distance=0.0,       # grains share a common plane
    merge_boundary_layer=True,    # remove duplicate boundary atoms
)

print(f"Sigma = {tilt_gb.info['sigma']}")
print(f"Disorientation = {tilt_gb.info['disorientation_angle']:.2f} deg")
print(f"Atoms = {len(tilt_gb)}")
print(f"Upper dirs = {tilt_gb.info['upper_dirs']}")
print(f"Lower dirs = {tilt_gb.info['lower_dirs']}")

### 3c. Auto-dispatch with `build_gb`

`build_gb` detects whether the rotation axis is [0001] (twist)
or in-plane (tilt) and calls the appropriate builder.

In [ ]:
# Twist — automatically detected
twist = build_gb(rec, element="Ti", a=a_Ti, c=c_Ti, n_layers=3)
print(f"Twist: {len(twist)} atoms")

# Tilt — automatically detected (using ideal c/a params from above)
tilt = build_gb(rec_tilt, element="Ti", a=a_ideal, c=c_ideal, n_layers=3)
print(f"Tilt:  {len(tilt)} atoms")

### 3d. Get slab directions for use with other GB builders

If you have an existing GB construction pipeline (e.g. `HCPGBGenerator`),
you can use `csl_slab_directions` to get the 4-index Miller-Bravais
direction triples.

In [ ]:
upper_dirs, lower_dirs = csl_slab_directions(rec_tilt, ca=ca_ideal)
print(f"Upper grain directions (4-index): {upper_dirs}")
print(f"Lower grain directions (4-index): {lower_dirs}")
print()
print("These can be passed to ASE's HexagonalClosedPacked(directions=...)")
print("or to HCPGBGenerator.from_basal_directions()")

## 4. Export structures

The returned `ase.Atoms` objects work with all ASE I/O formats.

In [ ]:
from ase.io import write

# POSCAR (VASP)
write("POSCAR_twist_S7", twist_gb, format="vasp")

# XYZ (for visualisation)
write("twist_S7.xyz", twist_gb)

print("Exported: POSCAR_twist_S7, twist_S7.xyz")

## 5. Single GB vs periodic cell

By default, `vacuum=0` gives a fully periodic cell with **2 GBs**
(one at the constructed interface, one at the periodic image).

Set `vacuum > 0` for a single GB with free surfaces.

In [ ]:
# Periodic cell (default): 2 GBs per cell
gb_periodic = build_gb(rec, element="Ti", a=a_Ti, c=c_Ti, n_layers=6)
print(f"Periodic:  {len(gb_periodic)} atoms, pbc={list(gb_periodic.pbc)}")
print(f"  Cell z = {gb_periodic.cell[2,2]:.1f} Å")

# Single GB: free surfaces at top and bottom
gb_single = build_gb(rec, element="Ti", a=a_Ti, c=c_Ti,
                     n_layers=6, vacuum=12.0)
print(f"\nSingle GB: {len(gb_single)} atoms, pbc={list(gb_single.pbc)}")
print(f"  Cell z = {gb_single.cell[2,2]:.1f} Å (includes 12 Å vacuum)")

## 6. Non-ideal c/a: enumerate at rational, rescale to real

Exact tilt CSLs require rational (c/a)².  For real metals, the
standard approach is:
1. Enumerate at a nearby rational c/a (e.g. sqrt(5/2) for Ti)
2. Build the bicrystal at that rational c/a
3. Rescale to the real lattice parameters

The misfit (~0.3% for Ti) is physically accommodated by DSC
dislocations at the boundary.

Common rational approximations (from Warrington 1975):
- **(c/a)² = 5/2** → Ti, Zr, Hf, Ru, Os, Y, Sc, Be
- **(c/a)² = 8/3** → Mg, Co, Re (ideal HCP)
- **(c/a)² = 7/2** → Zn, Cd

In [ ]:
# Ti: enumerate at (c/a)^2 = 5/2, then rescale to real lattice
ca_csl = sqrt(5 / 2)  # rational approximation
print(f"CSL c/a = {ca_csl:.4f} vs real Ti c/a = {c_Ti/a_Ti:.4f} "
      f"({(ca_csl - c_Ti/a_Ti)/(c_Ti/a_Ti)*100:+.2f}% strain)")

# Find a tilt GB that only exists at rational c/a
ti_res = enumerate_hcp_csl(ca_ratio=ca_csl, sigma_max=15, max_idx=5)
rec_ti_tilt = find_csl(ti_res, sigma=7,
                       axis_miller_bravais=[1, 0, -1, 0],
                       ca_ratio=ca_csl)[0]

# One-step: build at rational c/a and rescale to real Ti
gb_ti = build_gb_rescaled(
    rec_ti_tilt, element="Ti",
    a_real=a_Ti, c_real=c_Ti,
    ca_csl=ca_csl,
    n_layers=4,
)

print(f"\nSigma=7 [10-10] tilt for Ti:")
print(f"  Atoms: {len(gb_ti)}")
print(f"  Cell:  {gb_ti.cell.lengths().round(3)}")
print(f"  a-strain: {gb_ti.info['strain_a']*100:.3f}%")
print(f"  c-strain: {gb_ti.info['strain_c']*100:.3f}%")

## 7. Orthogonalize simulation cells

CSL supercells are often non-orthogonal.  Two tools are provided:

- **`orthogonalize_gb_plane(gb)`** — cheap integer shear that makes
  the stacking direction perpendicular to the GB plane (c ⊥ a, c ⊥ b).
  Same atom count.
- **`find_orthogonal_cell(gb)`** — full search for the smallest
  supercell with all angles ≈ 90°.  May increase atom count.

In [ ]:
import numpy as np

# --- Cheap: fix GB plane perpendicularity (tilt GBs) ---
# This removes the skew of the stacking vector via an integer shear.
# Same atom count, no supercell expansion.
gb_flat = orthogonalize_gb_plane(gb_ti)
print("orthogonalize_gb_plane (cheap skew removal):")
print(f"  Before: angles={np.round(gb_ti.cell.angles(), 1)}, {len(gb_ti)} atoms")
print(f"  After:  angles={np.round(gb_flat.cell.angles(), 1)}, {len(gb_flat)} atoms")
print(f"  Shear matrix P = {gb_flat.info.get('skew_correction', 'identity')}")

# --- Full: make all angles 90 deg (twist GBs) ---
# The hex CSL cell has 120° in-plane angle — this finds a rectangular supercell.
print("\nfind_orthogonal_cell (full orthogonalization):")
print(f"  Before: angles={np.round(twist_gb.cell.angles(), 1)}, {len(twist_gb)} atoms")
orth_twist = find_orthogonal_cell(twist_gb, max_search=8)
print(f"  After:  angles={np.round(orth_twist.cell.angles(), 1)}, {len(orth_twist)} atoms")
print(f"  Cell:   {np.round(orth_twist.cell.lengths(), 2)}")

## 8. Comparing materials

The [0001] twist CSLs are universal, but tilt GBs differ between materials.

In [ ]:
materials = {
    "Ti":       {"ca": 1.587, "a": 2.95,  "c": 4.68},
    "Mg":       {"ca": 1.624, "a": 3.21,  "c": 5.21},
    "Ideal":    {"ca": sqrt(8/3), "a": 1.0, "c": sqrt(8/3)},
}

for name, params in materials.items():
    res = enumerate_hcp_csl(ca_ratio=params["ca"], sigma_max=25, max_idx=3)
    sigmas = sorted(set(r["sigma"] for r in res))
    print(f"{name:>7} (c/a={params['ca']:.3f}): {len(res)} GBs, "
          f"Sigmas = {sigmas}")